
# Workflow Overview: ND2 to OME-Zarr, Colony & Nucleus Segmentation, Feature Extraction

The following set of notebook provides a reproducible workflow for high-content image analysis of timelpase live cell experiments. The workflow is designed to process microscopy data from raw ND2 files to quantitative feature extraction, enabling downstream biological analysis. The main steps are:

1. **ND2 to OME-Zarr conversion**: Convert raw ND2 microscopy files to the OME-Zarr format for scalable, cloud-ready storage and analysis.
2. **Colony segmentation using ConvPaint**: Identify and segment stem cell colonies in the images using a deep learning-based approach.
3. **Nucleus segmentation using StarDist / Cellpose**: Detect and segment individual nuclei within colonies for single-cell analysis.
4. **Cell Tracking**: Track individual cells over time to study dynamic behaviors.
5. **Feature Extraction**: Quantify spatial features and extract relevant biological markers (e.g., ERK, Oct4) for each cell.

Configuration options such as the output path or scaling parameters can be easily adjusted in the `configuration/settings.py` file. To change the way the dask cluster is started, modify the `configuration/dask.py` file. 

The workflow is highly modular, making it straightforward to adapt to different datasets or analysis needs. Once the ND2 files have been converted to OME-Zarr, the subsequent steps can be performed independently allowing you to skip or repeat steps as required for your analysis.


## 4. Cell Tracking using Trackastra
This notebook focuses on **cell tracking using Trackastra**. It can be seen as an alternative for ultra. 

Please ensure that the previous steps (ND2 to OME-Zarr conversion and colony segmentation) have been completed before running this notebook.

In [ ]:
from trackastra.model import Trackastra
import torch

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

model = Trackastra.from_pretrained("general_2d", device=device)


from configuration.settings import (
    get_output_path,
    get_fovs,
)  # Functions to get output path and list of FOVs
import configuration.settings as settings  # For additional settings (e.g., normalization axis)

tracking_version = 0  # Increment this when changing tracking parameters

INFO:trackastra.model.model:Loading model state from c:\Users\Niesen\Desktop\CellZarr\.venv\Lib\site-packages\trackastra\.models\general_2d\model.pt
INFO:trackastra.model.model_api:Using device cpu
INFO:trackastra.model.model_api:Default batch size = 1 for model on cpu.


c:\Users\Niesen\Desktop\CellZarr\.venv\Lib\site-packages\trackastra\.models\general_2d already downloaded, skipping.


In [ ]:
# Main tracking workflow: load data, segment nuclei, and save results
import ome_zarr.scale  # For scaling OME-Zarr data
import ome_zarr.reader as ozr  # For reading OME-Zarr data
import ome_zarr.io as ozi  # For OME-Zarr I/O operations
import ome_zarr.writer as ozw  # For writing label data to OME-Zarr
import dask.array as da  # For handling large arrays with Dask
import numpy as np  # For numerical operations
import zarr  # For Zarr storage
import os  # For file path operations
import tqdm  # For progress bars

from trackastra.tracking import graph_to_napari_tracks
import pickle
import gzip
import lzma


# Utility function to save label arrays to OME-Zarr format
def save_labels(label, label_name, root):
    # Remove existing label if present to avoid duplicates
    if "labels" in root:
        if label_name in root.labels.attrs["labels"]:
            del root["labels"][label_name]
            current_labels = root.labels.attrs["labels"]
            new_labels = [lbl for lbl in current_labels if lbl != label_name]
            root.labels.attrs["labels"] = new_labels
        try:
            del root["labels"][label_name]
        except:
            pass

    Y_dim = root["0"].shape[-2]
    X_dim = root["0"].shape[-1]
    # Write the label array to the OME-Zarr group
    return ozw.write_labels(
        labels=label,
        group=root,
        name=label_name,
        axes="tyx",
        scaler=ome_zarr.scale.Scaler(max_layer=1),
        chunks=(1, Y_dim, X_dim),
        storage_options={
            "compressor": zarr.storage.Blosc(cname="zstd", clevel=5),
        },
        metadata={"is_grayscale_label": False},
    )


# Function to process a single field of view (FOV)
def process_fov(fov):
    dest = os.path.join(get_output_path(), fov)  # Path to OME-Zarr data
    store = ozi.parse_url(dest, mode="a").store
    root = zarr.group(store=store)
    X_dim = root["0"].shape[-1]
    Y_dim = root["0"].shape[-2]
    nodes = list(ozr.Reader(ozi.parse_url(dest, mode="r"))())

    # Try to load colony mask, otherwise use all-ones mask
    try:
        i_colony = nodes[1].zarr.root_attrs["labels"].index("colony")
        colony = nodes[i_colony + 2].data[0]
    except ValueError:
        colony = da.ones((nodes[0].data.shape[0], Y_dim, X_dim), dtype=bool)

    raw = nodes[0].data[0]
    if "channel_names" in nodes[0].metadata:
        # Get H2B channel index from metadata
        H2B_channel = nodes[0].metadata["channel_names"].index("H2B")
    elif "channel" in nodes[0].metadata:
        # Get H2B channel index from metadata
        H2B_channel = nodes[0].metadata["name"].index("H2B")
    raw = raw[:, H2B_channel, :, :]  # Select H2B channel

    if "nucleus" in nodes[1].zarr.root_attrs["labels"]:
        i_nuc = nodes[1].zarr.root_attrs["labels"].index("nucleus")
        nuc_labels = nodes[i_nuc + 2].data[0]
    elif "nucleus_cellpose" in nodes[1].zarr.root_attrs["labels"]:
        i_nuc = nodes[1].zarr.root_attrs["labels"].index("nucleus_cellpose")
        nuc_labels = nodes[i_nuc + 2].data[0]
    else:
        assert "nucleus" not in nodes[1].zarr.root_attrs["labels"]

    raw = raw.compute()
    nuc_labels = nuc_labels.compute()
    track_graph, masks_tracked = model.track(raw, nuc_labels, mode="ilp")

    napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)

    with lzma.open(
        os.path.join(get_output_path(), f"{fov}_df_tracks_{tracking_version}.xz"), "wb"
    ) as f:
        pickle.dump(napari_tracks, f)

    with lzma.open(
        os.path.join(get_output_path(), f"{fov}_graph_{tracking_version}.xz"), "wb"
    ) as f:
        pickle.dump(napari_tracks_graph, f)

    save_labels(masks_tracked, "tracked", root)
    return


# Get list of FOVs to process
fovs = get_fovs()[0:1]
for fov in tqdm.tqdm(fovs, desc="Processing FOV", total=len(fovs)):
    process_fov(fov)

INFO:ome_zarr.reader:root_attr: multiscales
INFO:ome_zarr.reader:root_attr: omero
INFO:ome_zarr.reader:datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 1.0, 2.001044932079415, 2.001044932079415], 'type': 'scale'}], 'path': '1'}]
INFO:ome_zarr.reader:resolution: 0
INFO:ome_zarr.reader: - shape ('t', 'c', 'y', 'x') = (577, 3, 1915, 1915)
INFO:ome_zarr.reader: - chunks =  ['1', '1', '1915', '1915']
INFO:ome_zarr.reader: - dtype = uint16
INFO:ome_zarr.reader:resolution: 1
INFO:ome_zarr.reader: - shape ('t', 'c', 'y', 'x') = (577, 3, 957, 957)
INFO:ome_zarr.reader: - chunks =  ['1', '1', '957', '957']
INFO:ome_zarr.reader: - dtype = uint16
INFO:ome_zarr.reader:root_attr: labels
INFO:ome_zarr.reader:root_attr: image-label
INFO:ome_zarr.reader:root_attr: multiscales
INFO:ome_zarr.reader:root_attr: image-label
INFO:ome_zarr.reader:root_attr: multiscales
INFO:ome_zarr.reader:datasets [{'coord


Candidate graph		9145 nodes	8473 edges
Solution graph		9145 nodes	8165 edges


100%|██████████| 1346/1346 [00:00<00:00, 166923.90it/s]


[]